In [ ]:
!pip -q install transformers accelerate sentencepiece datasets evaluate scikit-learn pandas pyarrow biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 22.3 MB/s eta 0:00:00


In [ ]:
import torch, os
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


## milestone 0 : we imported our pretrained codonbert and tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM, AutoConfig
import torch

MODEL_ID = "lhallee/CodonBERT"

device = "cuda" if torch.cuda.is_available() else "cpu"

def try_load_hf(model_id: str):

    tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    try:
        model = AutoModelForMaskedLM.from_pretrained(model_id, trust_remote_code=True)
        head_type = "masked_lm"
    except Exception as e1:
        print("AutoModelForMaskedLM failed:", type(e1).__name__, e1)
        model = AutoModel.from_pretrained(model_id, trust_remote_code=True)
        head_type = "base"
    model.to(device)
    model.eval()
    return tok, model, head_type

tokenizer = None
model = None
head_type = None

try:
    tokenizer, model, head_type = try_load_hf(MODEL_ID)
    print("✅ Loaded from HF:", MODEL_ID, "| head:", head_type)
    print("Tokenizer vocab size:", getattr(tokenizer, "vocab_size", "NA"))
except Exception as e:
    print("❌ HF load failed:", type(e).__name__, e)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt:   0%|          | 0.00/356 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/348M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: lhallee/CodonBERT
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Loaded from HF: lhallee/CodonBERT | head: masked_lm
Tokenizer vocab size: 69


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()


codons = ["ATG"] + ["GCT"]*50 + ["TAA"]
text = " ".join(codons)

inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    out = model(**inputs)

print("input_ids shape:", inputs["input_ids"].shape)
print("logits shape:", out.logits.shape)

input_ids shape: torch.Size([1, 54])
logits shape: torch.Size([1, 54, 69])


In [ ]:
!pip -q install requests

In [ ]:
!pip -q install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 110.6 MB/s eta 0:00:00


## milestone 1 : downloaded yeast data set and added log expressions to it

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os, shutil, pathlib

DRIVE_ROOT = "/content/drive/MyDrive/codonbert_project"
CACHE = f"{DRIVE_ROOT}/cache"
LOCAL = "/content"

os.makedirs(CACHE, exist_ok=True)


for p in ["data", "outputs"]:
    lp = os.path.join(LOCAL, p)
    if os.path.exists(lp):
        shutil.rmtree(lp)
os.makedirs("data/expr/E-MTAB-8626", exist_ok=True)
os.makedirs("data/cds", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

print("Drive cache:", CACHE)

Drive cache: /content/drive/MyDrive/codonbert_project/cache


In [ ]:
!wget -O "{CACHE}/E-MTAB-8626-raw-counts.tsv.undecorated" \
  --tries=30 --waitretry=2 --timeout=30 --continue \
  ftp://ftp.ebi.ac.uk/pub/databases/microarray/data/atlas/experiments/E-MTAB-8626/E-MTAB-8626-raw-counts.tsv.undecorated

--2026-02-25 17:44:05--  ftp://ftp.ebi.ac.uk/pub/databases/microarray/data/atlas/experiments/E-MTAB-8626/E-MTAB-8626-raw-counts.tsv.undecorated
           => ‘/content/drive/MyDrive/codonbert_project/cache/E-MTAB-8626-raw-counts.tsv.undecorated’
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /pub/databases/microarray/data/atlas/experiments/E-MTAB-8626 ... done.
==> SIZE E-MTAB-8626-raw-counts.tsv.undecorated ... 792991
==> PASV ... done.    ==> RETR E-MTAB-8626-raw-counts.tsv.undecorated ... done.
Length: 792991 (774K) (unauthoritative)

E-MTAB-8626-raw-cou 100%[===================>] 774.41K   912KB/s    in 0.8s    

2026-02-25 17:44:08 (912 KB/s) - ‘/content/drive/MyDrive/codonbert_project/cache/E-MTAB-8626-raw-counts.tsv.undecorated’ saved [792991]



In [ ]:
!wget -O "{CACHE}/yeast_cds.fa.gz" \
  --tries=30 --waitretry=2 --timeout=30 --continue \
  https://ftp.ensemblgenomes.ebi.ac.uk/pub/fungi/release-56/fasta/saccharomyces_cerevisiae/cds/Saccharomyces_cerevisiae.R64-1-1.cds.all.fa.gz

--2026-02-25 17:44:25--  https://ftp.ensemblgenomes.ebi.ac.uk/pub/fungi/release-56/fasta/saccharomyces_cerevisiae/cds/Saccharomyces_cerevisiae.R64-1-1.cds.all.fa.gz
Resolving ftp.ensemblgenomes.ebi.ac.uk (ftp.ensemblgenomes.ebi.ac.uk)... 193.62.193.161
Connecting to ftp.ensemblgenomes.ebi.ac.uk (ftp.ensemblgenomes.ebi.ac.uk)|193.62.193.161|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3801332 (3.6M) [application/x-gzip]
Saving to: ‘/content/drive/MyDrive/codonbert_project/cache/yeast_cds.fa.gz’

/content/drive/MyDr 100%[===================>]   3.62M  2.91MB/s    in 1.2s    

2026-02-25 17:44:27 (2.91 MB/s) - ‘/content/drive/MyDrive/codonbert_project/cache/yeast_cds.fa.gz’ saved [3801332/3801332]



In [ ]:
!ls -lah "{CACHE}" | egrep "raw-counts|yeast_cds"
!head -n 2 "{CACHE}/E-MTAB-8626-raw-counts.tsv.undecorated"
!gunzip -c "{CACHE}/yeast_cds.fa.gz" | head -n 2

-rw------- 1 root root 775K Feb 25 17:44 E-MTAB-8626-raw-counts.tsv.undecorated
-rw------- 1 root root 3.7M Feb 16  2023 yeast_cds.fa.gz
Gene ID	ERR3773791	ERR3773792	ERR3773789	ERR3773777	ERR3773795	ERR3773783	ERR3773787	ERR3773794	ERR3773770	ERR3773780	ERR3773782	ERR3773772	ERR3773775	ERR3773773	ERR3773784	ERR3773785	ERR3773786	ERR3773788	ERR3773781	ERR3773790	ERR3773779	ERR3773769	ERR3773778	ERR3773771	ERR3773774	ERR3773776	ERR3773793
ETS1-1	2	11	3	2	2	3	21	2	4	5	1	2	6	1	1	2	9	1	2	10	0	0	2	2	1	3	2
>YPL071C_mRNA cds chromosome:R64-1-1:XVI:420048:420518:-1 gene:YPL071C gene_biotype:protein_coding transcript_biotype:protein_coding description:Putative protein of unknown function; green fluorescent protein (GFP)-fusion protein localizes to both the cytoplasm and the nucleus [Source:SGD;Acc:S000005992]
ATGAGTTCCCGGTTTGCAAGAAGTAATGGCAATCCCAACCACATTAGGAAAAGAAATCAT


In [ ]:
import os, shutil

DRIVE_ROOT = "/content/drive/MyDrive/codonbert_project"
CACHE = f"{DRIVE_ROOT}/cache"

os.makedirs("data/expr/E-MTAB-8626", exist_ok=True)
os.makedirs("data/cds", exist_ok=True)

shutil.copy(f"{CACHE}/E-MTAB-8626-raw-counts.tsv.undecorated",
            "data/expr/E-MTAB-8626/E-MTAB-8626-raw-counts.tsv.undecorated")
shutil.copy(f"{CACHE}/yeast_cds.fa.gz",
            "data/cds/yeast_cds.fa.gz")

print("Restored:",
      os.path.exists("data/expr/E-MTAB-8626/E-MTAB-8626-raw-counts.tsv.undecorated"),
      os.path.exists("data/cds/yeast_cds.fa.gz"))

Restored: True True


In [ ]:
import pandas as pd
import numpy as np

raw_path = "data/expr/E-MTAB-8626/E-MTAB-8626-raw-counts.tsv.undecorated"
raw = pd.read_csv(raw_path, sep="\t")

print("raw shape:", raw.shape)
print("first cols:", list(raw.columns[:15]))

# gene column detection
gene_col = None
for c in raw.columns:
    if c.lower() in ["geneid", "gene_id", "gene id", "gene"]:
        gene_col = c
        break
if gene_col is None:
    gene_col = raw.columns[0]

sample_cols = [c for c in raw.columns if c != gene_col]
num = raw[sample_cols].apply(pd.to_numeric, errors="coerce")

expr = pd.DataFrame({
    "gene_id": raw[gene_col].astype(str),
    "expression_raw": num.median(axis=1, skipna=True)
}).dropna()

expr = expr.groupby("gene_id", as_index=False)["expression_raw"].median()

print("expr shape:", expr.shape)
expr.head()

raw shape: (7127, 28)
first cols: ['Gene ID', 'ERR3773791', 'ERR3773792', 'ERR3773789', 'ERR3773777', 'ERR3773795', 'ERR3773783', 'ERR3773787', 'ERR3773794', 'ERR3773770', 'ERR3773780', 'ERR3773782', 'ERR3773772', 'ERR3773775', 'ERR3773773']
expr shape: (7127, 2)


,gene_id,expression_raw
0,ETS1-1,2.0
1,ETS1-2,0.0
2,ETS2-1,0.0
3,ETS2-2,0.0
4,HRA1,8.0


In [ ]:
import gzip, re
import pandas as pd

STOP_CODONS_DNA = {"TAA", "TAG", "TGA"}
DNA_OK = set("ACGT")

def extract_gene(desc: str):
    m = re.search(r"gene:([A-Za-z0-9_]+)", desc)
    return m.group(1) if m else desc.split()[0]

def clean_cds_dna(seq: str):
    s = seq.strip().upper()
    s = "".join(ch if ch in DNA_OK else "U" for ch in s)
    if len(s) < 6: return None, False, "too_short"
    if not s.startswith("ATG"): return None, False, "no_start_ATG"
    if len(s) % 3 != 0: return None, False, "len_not_multiple_of_3"
    if s[-3:] not in STOP_CODONS_DNA: return None, False, "no_stop_codon"
    return s, True, "ok"

def dna_to_rna(s: str):
    return s.replace("T", "U")

fasta_path = "data/cds/yeast_cds.fa.gz"

rows = []
reasons = {}
total = kept = 0

with gzip.open(fasta_path, "rt") as f:
    header = None
    seq_chunks = []

    for line in f:
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            # flush previous
            if header is not None:
                total += 1
                gid = extract_gene(header)
                dna = "".join(seq_chunks)
                dna_clean, ok, reason = clean_cds_dna(dna)
                reasons[reason] = reasons.get(reason, 0) + 1
                if ok:
                    kept += 1
                    rows.append({
                        "gene_id": gid,
                        "cds_dna_clean": dna_clean,
                        "cds_rna_clean": dna_to_rna(dna_clean),
                        "len_nt": len(dna_clean),
                        "len_codons": len(dna_clean)//3
                    })
            header = line[1:]
            seq_chunks = []
        else:
            seq_chunks.append(line)

    # last record
    if header is not None:
        total += 1
        gid = extract_gene(header)
        dna = "".join(seq_chunks)
        dna_clean, ok, reason = clean_cds_dna(dna)
        reasons[reason] = reasons.get(reason, 0) + 1
        if ok:
            kept += 1
            rows.append({
                "gene_id": gid,
                "cds_dna_clean": dna_clean,
                "cds_rna_clean": dna_to_rna(dna_clean),
                "len_nt": len(dna_clean),
                "len_codons": len(dna_clean)//3
            })

cds_df = pd.DataFrame(rows)

print("CDS total:", total, "kept:", kept)
print("filter reasons:", reasons)
cds_df.head()

CDS total: 6600 kept: 6590
filter reasons: {'ok': 6590, 'no_start_ATG': 10}


,gene_id,cds_dna_clean,cds_rna_clean,len_nt,len_codons
0,YPL071C,ATGAGTTCCCGGTTTGCAAGAAGTAATGGCAATCCCAACCACATTA...,AUGAGUUCCCGGUUUGCAAGAAGUAAUGGCAAUCCCAACCACAUUA...,471,157
1,YLL050C,ATGTCTAGATCTGGTGTTGCTGTTGCTGATGAATCCCTTACCGCTT...,AUGUCUAGAUCUGGUGUUGCUGUUGCUGAUGAAUCCCUUACCGCUU...,432,144
2,YMR172W,ATGTCTGGAATGGGTATTGCGATTCTTTGCATCGTACGTACAAAGA...,AUGUCUGGAAUGGGUAUUGCGAUUCUUUGCAUCGUACGUACAAAGA...,2160,720
3,YOR185C,ATGTCAGCACCTGCTCAAAACAATGCCGAGGTTCCCACTTTCAAGT...,AUGUCAGCACCUGCUCAAAACAAUGCCGAGGUUCCCACUUUCAAGU...,663,221
4,YLL032C,ATGGATAACTTCAAAATTTACAGTACAGTTATCACAACTGCTTTTT...,AUGGAUAACUUCAAAAUUUACAGUACAGUUAUCACAACUGCUUUUU...,2478,826


In [ ]:
import os
import pandas as pd

merged = cds_df.merge(expr, on="gene_id", how="inner").drop_duplicates("gene_id")

print("merged genes:", merged.shape[0])
print(merged[["gene_id","len_codons","expression_raw"]].head())

# save local
out_local = "outputs/yeast_m1_merged.parquet"
merged.to_parquet(out_local, index=False)
print("saved local:", out_local)

# save to drive
out_drive_dir = f"{DRIVE_ROOT}/outputs"
os.makedirs(out_drive_dir, exist_ok=True)
out_drive = f"{out_drive_dir}/yeast_m1_merged.parquet"
merged.to_parquet(out_drive, index=False)
print("saved drive:", out_drive)

merged genes: 6030
   gene_id  len_codons  expression_raw
0  YPL071C         157           201.0
1  YLL050C         144          3331.0
2  YMR172W         720           454.0
3  YOR185C         221           367.0
4  YLL032C         826           368.0
saved local: outputs/yeast_m1_merged.parquet
saved drive: /content/drive/MyDrive/codonbert_project/outputs/yeast_m1_merged.parquet


## milestone 2: adding labels : low, mid, high for training our classifier

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_parquet("outputs/yeast_m1_merged.parquet")

df["expr_log"] = np.log1p(df["expression_raw"])

q33 = df["expr_log"].quantile(0.33)
q67 = df["expr_log"].quantile(0.67)

df["label"] = np.where(df["expr_log"] < q33, 0,
                np.where(df["expr_log"] < q67, 1, 2)).astype(int)

print("q33:", q33, "q67:", q67)
print("label counts:\n", df["label"].value_counts().sort_index())

df.to_parquet("outputs/yeast_m2_labeled.parquet", index=False)
print("saved -> outputs/yeast_m2_labeled.parquet")


drive_path = "/content/drive/MyDrive/codonbert_project/outputs/yeast_m2_labeled.parquet"
df.to_parquet(drive_path, index=False)
print("saved ->", drive_path)

q33: 5.697093486505405 q67: 6.654152520183219
label counts:
 label
0    1987
1    2049
2    1994
Name: count, dtype: int64
saved -> outputs/yeast_m2_labeled.parquet
saved -> /content/drive/MyDrive/codonbert_project/outputs/yeast_m2_labeled.parquet


## milestone 3 : getting bert input ready with tokenization

In [ ]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer

MODEL_ID = "lhallee/CodonBERT"
MAX_LEN = 1064

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, do_lower_case=False)

df = pd.read_parquet("outputs/yeast_m2_labeled.parquet")

def rna_to_spaced_codons(rna: str):

    return " ".join([rna[i:i+3] for i in range(0, len(rna), 3)])


df["text"] = df["cds_rna_clean"].apply(rna_to_spaced_codons)

# tokenize
enc = tokenizer(
    df["text"].tolist(),
    padding="max_length",
    truncation=True,
    max_length=MAX_LEN,
    return_attention_mask=True
)

df["input_ids"] = enc["input_ids"]
df["attention_mask"] = enc["attention_mask"]

# sanity checks
print("Example tokenized length:", len(df["input_ids"].iloc[0]))
print("UNK count in sample:", df["input_ids"].iloc[0].count(tokenizer.unk_token_id))
print("MAX_LEN:", MAX_LEN)

out_local = "outputs/yeast_m3_encoded.parquet"
df.drop(columns=["text"]).to_parquet(out_local, index=False)
print("saved ->", out_local)

out_drive = "/content/drive/MyDrive/codonbert_project/outputs/yeast_m3_encoded.parquet"
df.drop(columns=["text"]).to_parquet(out_drive, index=False)
print("saved ->", out_drive)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt:   0%|          | 0.00/356 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Example tokenized length: 1064
UNK count in sample: 0
MAX_LEN: 1064
saved -> outputs/yeast_m3_encoded.parquet
saved -> /content/drive/MyDrive/codonbert_project/outputs/yeast_m3_encoded.parquet


## milestone 4 : train our classifier

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_parquet("/content/drive/MyDrive/codonbert_project/outputs/yeast_m3_encoded.parquet")

df = df[["gene_id","input_ids","attention_mask","label","expression_raw"]].copy()


train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df["label"]
)

print("train:", train_df.shape, "val:", val_df.shape)
print("train label counts:\n", train_df["label"].value_counts().sort_index())
print("val label counts:\n", val_df["label"].value_counts().sort_index())

train: (5125, 5) val: (905, 5)
train label counts:
 label
0    1689
1    1741
2    1695
Name: count, dtype: int64
val label counts:
 label
0    298
1    308
2    299
Name: count, dtype: int64


In [ ]:
import torch
from torch.utils.data import Dataset

MAX_LEN = 1024

class CodonClsDataset(Dataset):
    def __init__(self, frame):
        self.input_ids = frame["input_ids"].tolist()
        self.attn = frame["attention_mask"].tolist()
        self.labels = frame["label"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        ids = self.input_ids[idx][:MAX_LEN]
        am  = self.attn[idx][:MAX_LEN]
        return {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "attention_mask": torch.tensor(am, dtype=torch.long),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }

train_ds = CodonClsDataset(train_df)
val_ds   = CodonClsDataset(val_df)

print("train_ds:", len(train_ds), "val_ds:", len(val_ds))

train_ds: 5125 val_ds: 905


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from transformers import AutoConfig, AutoModel
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

MODEL_ID = "lhallee/CodonBERT"
num_labels = 3

config = AutoConfig.from_pretrained(MODEL_ID, trust_remote_code=True)
backbone = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True)

class CodonClassifier(nn.Module):
    def __init__(self, backbone, hidden_size, num_labels):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, labels=None):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls))
        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)
        return {"loss": loss, "logits": logits}

model = CodonClassifier(backbone, config.hidden_size, num_labels)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    acc = accuracy_score(labels, preds)

    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )

    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    try:
        auroc = roc_auc_score(labels, probs, multi_class="ovr")
    except Exception:
        auroc = float("nan")

    return {
        "accuracy": acc,
        "precision_macro": prec_m,
        "recall_macro": rec_m,
        "f1_macro": f1_m,
        "precision_weighted": prec_w,
        "recall_weighted": rec_w,
        "f1_weighted": f1_w,
        "auroc_ovr": auroc,
    }

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: lhallee/CodonBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
OUT_DIR = "/content/drive/MyDrive/codonbert_project/checkpoints/yeast_classifier"
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", OUT_DIR, "exists?", os.path.exists(OUT_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OUT_DIR: /content/drive/MyDrive/codonbert_project/checkpoints/yeast_classifier exists? True


In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir=OUT_DIR,

    eval_strategy="steps",
    eval_steps=200,

    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=2e-5,
    weight_decay=0.01,

    fp16=True,
    logging_steps=50,
    report_to="none",
)

In [ ]:
import os, torch

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

trainer.train()


manual_path = os.path.join(OUT_DIR, "final_state_dict.pt")
torch.save(model.state_dict(), manual_path)
print("Saved manual state_dict:", manual_path)

print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Final eval:", trainer.evaluate(val_ds))

Step,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted,Auroc Ovr
200,0.776636,0.771044,0.655249,0.673284,0.655701,0.653169,0.671947,0.655249,0.652393,0.834759
400,0.680879,0.786278,0.658564,0.659120,0.659531,0.658896,0.658110,0.658564,0.657912,0.836820
600,0.645983,0.777632,0.666298,0.663874,0.667640,0.665333,0.662941,0.666298,0.664193,0.836543
800,0.532144,0.825586,0.665193,0.669864,0.665670,0.667272,0.668866,0.665193,0.666531,0.831092


Saved manual state_dict: /content/drive/MyDrive/codonbert_project/checkpoints/yeast_classifier/final_state_dict.pt
Best checkpoint: /content/drive/MyDrive/codonbert_project/checkpoints/yeast_classifier/checkpoint-800


Final eval: {'eval_loss': 0.825586199760437, 'eval_accuracy': 0.6651933701657459, 'eval_precision_macro': 0.6698642196188423, 'eval_recall_macro': 0.6656697548147796, 'eval_f1_macro': 0.6672718195986752, 'eval_precision_weighted': 0.668866046349997, 'eval_recall_weighted': 0.6651933701657459, 'eval_f1_weighted': 0.6665311713934212, 'eval_auroc_ovr': 0.83109218468025, 'eval_runtime': 18.0698, 'eval_samples_per_second': 50.083, 'eval_steps_per_second': 25.069, 'epoch': 3.0}


In [ ]:
import glob
print("Checkpoints on Drive:")
print("\n".join(glob.glob(OUT_DIR + "/checkpoint-*")[:10]))
print("manual exists?", os.path.exists(OUT_DIR + "/final_state_dict.pt"))

Checkpoints on Drive:
/content/drive/MyDrive/codonbert_project/checkpoints/yeast_classifier/checkpoint-600
/content/drive/MyDrive/codonbert_project/checkpoints/yeast_classifier/checkpoint-800
/content/drive/MyDrive/codonbert_project/checkpoints/yeast_classifier/checkpoint-963
manual exists? True


### now we have our classifier lets see the metrics :

In [ ]:
import numpy as np
import torch
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, confusion_matrix, classification_report
)

pred = trainer.predict(val_ds)
logits = pred.predictions
y_true = pred.label_ids
y_pred = np.argmax(logits, axis=1)

acc = accuracy_score(y_true, y_pred)

prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)

probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
auroc = roc_auc_score(y_true, probs, multi_class="ovr")

print("VAL accuracy:", acc)
print("VAL precision_macro:", prec_m)
print("VAL recall_macro:", rec_m)
print("VAL f1_macro:", f1_m)
print("VAL precision_weighted:", prec_w)
print("VAL recall_weighted:", rec_w)
print("VAL f1_weighted:", f1_w)
print("VAL AUROC (OVR):", auroc)

print("\nConfusion matrix:\n", confusion_matrix(y_true, y_pred))
print("\nPer-class report:\n", classification_report(y_true, y_pred, digits=4))

VAL accuracy: 0.6651933701657459
VAL precision_macro: 0.6698642196188423
VAL recall_macro: 0.6656697548147796
VAL f1_macro: 0.6672718195986752
VAL precision_weighted: 0.668866046349997
VAL recall_weighted: 0.6651933701657459
VAL f1_weighted: 0.6665311713934212
VAL AUROC (OVR): 0.83109218468025

Confusion matrix:
 [[196  81  21]
 [ 64 190  54]
 [ 22  61 216]]

Per-class report:
               precision    recall  f1-score   support

           0     0.6950    0.6577    0.6759       298
           1     0.5723    0.6169    0.5938       308
           2     0.7423    0.7224    0.7322       299

    accuracy                         0.6652       905
   macro avg     0.6699    0.6657    0.6673       905
weighted avg     0.6689    0.6652    0.6665       905



## milestone 5 : fine tuning for regression model

In [ ]:
!pip -q install scipy fastparquet

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

MAX_LEN = 1024
DATA_PATH = "/content/drive/MyDrive/codonbert_project/outputs/yeast_m3_encoded.parquet"

df = pd.read_parquet(DATA_PATH, engine="fastparquet")


df["y_log"] = np.log1p(df["expression_raw"])
y_min, y_max = df["y_log"].min(), df["y_log"].max()
df["y_scaled"] = 2.0 * (df["y_log"] - y_min) / (y_max - y_min) - 1.0

# split: 70/15/15
trainval_df, test_df = train_test_split(df, test_size=0.15, random_state=42)
train_df, val_df = train_test_split(trainval_df, test_size=0.1765, random_state=42)  # 0.1765*0.85≈0.15

print("train/val/test:", len(train_df), len(val_df), len(test_df))
print("y_scaled range:", (df["y_scaled"].min(), df["y_scaled"].max()))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.1 MB/s eta 0:00:00
train/val/test: 4220 905 905
y_scaled range: (-1.0, 1.0)


In [ ]:
import torch
from torch.utils.data import Dataset

class CodonRegDataset(Dataset):
    def __init__(self, frame):
        self.input_ids = frame["input_ids"].tolist()
        self.attn = frame["attention_mask"].tolist()
        self.y = frame["y_scaled"].tolist()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return {
            "input_ids": torch.tensor(self.input_ids[idx][:MAX_LEN], dtype=torch.long),
            "attention_mask": torch.tensor(self.attn[idx][:MAX_LEN], dtype=torch.long),
            "labels": torch.tensor(self.y[idx], dtype=torch.float),   # Trainer expects labels
        }

train_reg_ds = CodonRegDataset(train_df)
val_reg_ds   = CodonRegDataset(val_df)
test_reg_ds  = CodonRegDataset(test_df)

In [ ]:
import torch.nn as nn


backbone = trainer.model.backbone

class CodonRegressor(nn.Module):
    def __init__(self, backbone, hidden_size):
        super().__init__()
        self.backbone = backbone
        self.dropout = nn.Dropout(0.1)
        self.regressor = nn.Linear(hidden_size, 1)
    def forward(self, input_ids, attention_mask, labels=None):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        pred = self.regressor(self.dropout(cls)).squeeze(-1)
        loss = None
        if labels is not None:
            loss = nn.MSELoss()(pred, labels)
        return {"loss": loss, "logits": pred}

reg_model = CodonRegressor(backbone, trainer.model.classifier.in_features)

# freeze first 6 layers
for i in range(6):
    for p in reg_model.backbone.encoder.layer[i].parameters():
        p.requires_grad = False

trainable = sum(p.numel() for p in reg_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in reg_model.parameters())
print(f"Trainable: {trainable/1e6:.2f}M / Total: {total/1e6:.2f}M")

Trainable: 43.96M / Total: 86.49M


In [ ]:
import os, torch
from transformers import Trainer, TrainingArguments
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr
import numpy as np

OUT_DIR = "/content/drive/MyDrive/codonbert_project/checkpoints/yeast_regression"
os.makedirs(OUT_DIR, exist_ok=True)

def compute_reg_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.array(preds).reshape(-1)
    labels = np.array(labels).reshape(-1)
    mse = mean_squared_error(labels, preds)
    mae = mean_absolute_error(labels, preds)
    rho, _ = spearmanr(labels, preds)
    return {"mse": mse, "mae": mae, "spearman": float(rho)}

args_reg = TrainingArguments(
    output_dir=OUT_DIR,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="spearman",
    greater_is_better=True,
    num_train_epochs=5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=True,
    logging_steps=50,
    report_to="none",
)

reg_trainer = Trainer(
    model=reg_model,
    args=args_reg,
    train_dataset=train_reg_ds,
    eval_dataset=val_reg_ds,
    compute_metrics=compute_reg_metrics,
)

reg_trainer.train()

manual_path = os.path.join(OUT_DIR, "final_reg_state_dict.pt")
torch.save(reg_model.state_dict(), manual_path)
print("Saved manual:", manual_path)
print("Best checkpoint:", reg_trainer.state.best_model_checkpoint)

Step,Training Loss,Validation Loss,Mse,Mae,Spearman
200,0.045936,0.041175,0.041175,0.136377,0.808702
400,0.035352,0.036853,0.036853,0.126010,0.810564
600,0.031896,0.039120,0.039120,0.140103,0.815314
800,0.029651,0.035010,0.035010,0.122453,0.811098
1000,0.028341,0.033909,0.033909,0.121934,0.825821
1200,0.022940,0.035349,0.035349,0.123686,0.818281


Saved manual: /content/drive/MyDrive/codonbert_project/checkpoints/yeast_regression/final_reg_state_dict.pt
Best checkpoint: /content/drive/MyDrive/codonbert_project/checkpoints/yeast_regression/checkpoint-1000


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import spearmanr


pred = reg_trainer.predict(test_reg_ds)
y_pred_scaled = np.array(pred.predictions).reshape(-1)
y_true_scaled = np.array(pred.label_ids).reshape(-1)


mse = mean_squared_error(y_true_scaled, y_pred_scaled)
mae = mean_absolute_error(y_true_scaled, y_pred_scaled)
rho_scaled, _ = spearmanr(y_true_scaled, y_pred_scaled)

print("TEST (scaled) MSE:", mse)
print("TEST (scaled) MAE:", mae)
print("TEST (scaled) Spearman:", float(rho_scaled))


def inv_scale(y_s, y_min, y_max):
    return ((y_s + 1.0) / 2.0) * (y_max - y_min) + y_min

y_pred_log = inv_scale(y_pred_scaled, y_min, y_max)
y_true_log = inv_scale(y_true_scaled, y_min, y_max)

y_pred_raw = np.expm1(y_pred_log)
y_true_raw = np.expm1(y_true_log)

rho_raw, _ = spearmanr(y_true_raw, y_pred_raw)
print("TEST Spearman on raw expression:", float(rho_raw))

out_path = "/content/drive/MyDrive/codonbert_project/outputs/test_predictions.csv"

if "test_df" in globals() and "gene_id" in test_df.columns:
    out = pd.DataFrame({
        "gene_id": test_df["gene_id"].values,
        "true_expression": y_true_raw,
        "pred_expression": y_pred_raw
    })
else:
    out = pd.DataFrame({
        "true_expression": y_true_raw,
        "pred_expression": y_pred_raw
    })

out.to_csv(out_path, index=False)
print("Saved:", out_path)
out.head()

TEST (scaled) MSE: 0.028442492708563805
TEST (scaled) MAE: 0.11420632153749466
TEST (scaled) Spearman: 0.8631951899009941
TEST Spearman on raw expression: 0.8631951899009941
Saved: /content/drive/MyDrive/codonbert_project/outputs/test_predictions.csv


,gene_id,true_expression,pred_expression
0,YCL046W,1.000000,33.424400
1,YPR060C,618.999695,920.851013
2,YHR024C,1501.000610,1593.130127
3,YHR053C,0.000000,108.030899
4,YOR354C,1413.999878,546.071350


In [ ]:
import numpy as np

pred = reg_trainer.predict(test_reg_ds)
y_pred_scaled = np.array(pred.predictions).reshape(-1)
y_true_scaled = np.array(pred.label_ids).reshape(-1)

print("y_true_scaled range:", y_true_scaled.min(), y_true_scaled.max())
print("y_pred_scaled range:", y_pred_scaled.min(), y_pred_scaled.max())

y_true_scaled range: -1.0 0.918913
y_pred_scaled range: -1.0146484 0.69091797


In [ ]:
import numpy as np
import pandas as pd


pred = reg_trainer.predict(test_reg_ds)
pred_scaled = np.array(pred.predictions).reshape(-1)
true_scaled = np.array(pred.label_ids).reshape(-1)


def inv_scale(y_s, y_min, y_max):
    return ((y_s + 1.0) / 2.0) * (y_max - y_min) + y_min

pred_log = inv_scale(pred_scaled, y_min, y_max)
true_log = inv_scale(true_scaled, y_min, y_max)

pred_raw = np.expm1(pred_log)
true_raw = np.expm1(true_log)


out = pd.DataFrame({
    "gene_id": test_df["gene_id"].values if "test_df" in globals() and "gene_id" in test_df.columns else np.arange(len(true_scaled)),
    "true_scaled": true_scaled,
    "pred_scaled": pred_scaled,
    "true_expression": true_raw,
    "pred_expression": pred_raw
})

out.head()

,gene_id,true_scaled,pred_scaled,true_expression,pred_expression
0,YCL046W,-0.880736,-0.391113,1.000000,33.424400
1,YPR060C,0.106310,0.174561,618.999695,920.851013
2,YHR024C,0.258556,0.268799,1501.000610,1593.130127
3,YHR053C,-1.000000,-0.192749,0.000000,108.030899
4,YOR354C,0.248289,0.084778,1413.999878,546.071350


In [ ]:
out_path = "/content/drive/MyDrive/codonbert_project/outputs/test_predictions_scaled_and_raw.csv"
out.to_csv(out_path, index=False)
print("Saved:", out_path)

Saved: /content/drive/MyDrive/codonbert_project/outputs/test_predictions_scaled_and_raw.csv
